Data Cleaning

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Read the original dataset
df = pd.read_csv("rawdata.csv")

print("Original Shape:", df.shape)

print("\nMissing Values Before Cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nDuplicate Rows:", df.duplicated().sum())


# 2. Create a copy for cleaning
df_clean = df.copy()


# 3. Handle missing values

# Fill missing Children values with median
df_clean["Children"] = df_clean["Children"].fillna(
    df_clean["Children"].median()
).astype(int)

# Fill missing Country values with Unknown
df_clean["Country"] = df_clean["Country"].fillna("Unknown")


# 4. Convert date column to datetime
df_clean["ReservationStatusDate"] = pd.to_datetime(
    df_clean["ReservationStatusDate"],
    errors="coerce"
)


# 5. Remove extra spaces from text columns
for col in df_clean.select_dtypes(
    include=["object", "str"]
).columns:
    df_clean[col] = df_clean[col].str.strip()


# 6. Check duplicate rows
duplicates = df_clean.duplicated().sum()

print("\nDuplicate Rows Found:", duplicates)

# Keep duplicates for now because identical
# booking records may represent different bookings.


# 7. Check invalid or unusual values

# Negative ADR
negative_adr = (df_clean["ADR"] < 0).sum()

print("\nNegative ADR:", negative_adr)

# Zero guests
zero_guests = (
    (df_clean["Adults"] == 0) &
    (df_clean["Children"] == 0) &
    (df_clean["Babies"] == 0)
)

print("Zero Guests:", zero_guests.sum())

# Keep unusual records for investigation.
# Do not delete them automatically.


# 8. Check results after cleaning
print("\nCleaned Shape:", df_clean.shape)

print("\nRemaining Missing Values:")
print(
    df_clean.isnull().sum()[
        df_clean.isnull().sum() > 0
    ]
)

print("\nFirst 5 Rows:")
display(df_clean.head())


# 9. Save cleaned dataset
output_file = Path("cleaned_H2.csv")

try:
    df_clean.to_csv(output_file, index=False)

except PermissionError:
    print("\nOriginal CSV is locked.")
    output_file = Path("cleaned_H2_v2.csv")
    df_clean.to_csv(output_file, index=False)


# 10. Final message
print("\nInitial cleaning completed!")
print("Dataset saved as:", output_file.resolve())
print("Final Shape:", df_clean.shape)

Original Shape: (79330, 31)

Missing Values Before Cleaning:
Children     4
Country     24
dtype: int64

Duplicate Rows: 25902

Duplicate Rows Found: 25902

Negative ADR: 0
Zero Guests: 167

Cleaned Shape: (79330, 31)

Remaining Missing Values:
Series([], dtype: int64)

First 5 Rows:


,IsCanceled,LeadTime,ArrivalDateYear,ArrivalDateMonth,ArrivalDateWeekNumber,ArrivalDateDayOfMonth,StaysInWeekendNights,StaysInWeekNights,Adults,Children,...,DepositType,Agent,Company,DaysInWaitingList,CustomerType,ADR,RequiredCarParkingSpaces,TotalOfSpecialRequests,ReservationStatus,ReservationStatusDate
0,0,6,2015,July,27,1,0,2,1,0,...,No Deposit,6,NULL,0,Transient,0.0,0,0,Check-Out,2015-07-03
1,1,88,2015,July,27,1,0,4,2,0,...,No Deposit,9,NULL,0,Transient,76.5,0,1,Canceled,2015-07-01
2,1,65,2015,July,27,1,0,4,1,0,...,No Deposit,9,NULL,0,Transient,68.0,0,1,Canceled,2015-04-30
3,1,92,2015,July,27,1,2,4,2,0,...,No Deposit,9,NULL,0,Transient,76.5,0,2,Canceled,2015-06-23
4,1,100,2015,July,27,2,0,2,2,0,...,No Deposit,9,NULL,0,Transient,76.5,0,1,Canceled,2015-04-02



Initial cleaning completed!
Dataset saved as: C:\Users\omarm\OneDrive\Desktop\ML Project\ML-Project\cleaned_H2.csv
Final Shape: (79330, 31)
